# Chapter 12 Companion Notebook: Clustering: Bank Customer K-Modes

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch12_Clustering_Bank_Customer_K_Modes.ipynb)

This notebook accompanies Chapter 12 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
- Click "Upload" and select this file and the data file.
- https://archive.ics.uci.edu/dataset/222/bank+marketing

# Bank customer: K-modes clustering (Categorical variables only)

### Use "Bank customer.csv"
Build a classification algorithm for bank customers' response to marketing campaign.
- age (numeric)  
- marital: marital status (categorical: "married", "divorced", "single"; "divorced" means divorced or widowed)  - education (categorical: "secondary", "primary", "tertiary")  
- default: has credit in default? (binary: "yes", "no")  
- balance: average yearly balance, in euros (numeric)  
- housing: has housing loan? (binary: "yes", "no")  
- loan: has personal loan? (binary: "yes", "no")  
- duration: last contact duration, in seconds (numeric)  
- campaign: number of contacts performed during this campaign (numeric, includes last contact)  
- pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric) 
- previous: number of contacts performed before this campaign (numeric)  
- poutcome: outcome of the previous marketing campaign (categorical: "failure", "success") 
- deposit: has the client subscribed a term deposit? (binary: "yes", "no")

### Install kmodes

In [ ]:
# pip install kmodes

In [ ]:
import numpy as np
import pandas as pd
from kmodes.kmodes import KModes
from kmodes.kprototypes import KPrototypes

### # Create dummy variables for categorical columns and save dummy variables only except 'Deposit'.

In [ ]:
df = pd.read_csv('Bank customer.csv')
df.head()

In [ ]:
# Create dummy variables for categorical columns

df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
df_cat=df.iloc[:, 6:-1]  # all rows and column 4~6. same as [:, 4:]
df_cat.head()

### 2. Conduct K-modes clustering using categorical variables

In [ ]:
m = KModes(n_clusters=4, init='Huang', n_init=5, verbose=1)
cls = m.fit_predict(df_cat)

- `KModes(n_clusters=4, init='Huang', n_init=5, verbose=1)`: This initializes a k-modes clustering model
    - `n_clusters=4` specifies that the algorithm should form four clusters.
    - `init='Huang'` sets the initialization method to 'Huang', which is often more efficient for categorical data than random initialization.
    - `n_init=5` indicates that the k-modes algorithm will run five times with different centroid seeds, and the best run in terms of cost (error) will be chosen.
    - `verbose=1` will output logs about the clustering process (useful for debugging or understanding the process).
- `cls = m.fit_predict(df_cat)`: fits the model using the categorical features stored in df_cat and assigns each data point to one of the four clusters.
    - `cls`: stores the cluster labels for each data point

In [ ]:
m.cluster_centroids_ 

- `m.cluster_centroids_`: This attribute of the fitted k-modes model m contains the centroids of the formed clusters. Each centroid is represented by the point in the feature space that corresponds to the mode of all points in that cluster.
    - [n_clusters, n_features]=(4,3)

In [ ]:
m.cost_

- `m.cost_`: This attribute of the fitted k-modes model m represents the cost, or clustering error, of the final clustering solution. The cost is computed as the sum of the dissimilarities (distance) between each sample and its corresponding cluster mode (centroid).

# K-prototypes clustering for mixed categorical and numerical variables

### Drop 'deposit_yes' and save the variables as array.

In [ ]:
# Drop the 'deposit_yes' column
df = df.drop(columns=['deposit_yes'])
df.head()

In [ ]:
array = df.values  # save df as array
array

- `array = df.values`: converts the DataFrame into a NumPy array. 
- list vs. array: List can contains different data types. Array can contains same data types

### 2. Conduct K-protopypes clustering using all variables.

In [ ]:
# Select dummy variable columns from 'array' using indices

cat = list(range(6, array.shape[1]))  # Extract column indices from position 6 onward
kproto = KPrototypes(n_clusters=3, verbose=2, max_iter=20).fit(array, categorical=cat)

### 3. Display cluster centers

In [ ]:
kproto.cluster_centroids_  

- `kproto.cluster_centroids_`: The centroids of the clusters formed.
- [n_clusters, n_variables]

### 4. Preict clusters for the samples. Display samples that belong to Cluster=1

In [ ]:
# Prediction

cls=kproto.predict(array, categorical=cat)
df['cluster'] = cls
df.head()
# df['cluster'] = kproto.labels_  # alternative

- `cls = kproto.predict(array, categorical=cat)`: This line predicts the closest cluster for each sample
- `df['cluster'] = cls` : This line adds a new column to the original DataFrame, labeled 'cluster', which contains the cluster assignments for each row. 
- `df['cluster'] = kproto.labels_` is an alternative way to achieve the same result without calling the predict method. After fitting a k-prototypes model, the labels_ attribute contains the cluster labels for each point.

In [ ]:
df[df.cluster==1].head()

- `df['cluster'] == 1`: creates a Boolean series. It goes through the 'cluster' column and checks each row to see whether its value is equal to 1. If a row has the value 1, the corresponding entry in the Boolean series is True; otherwise, it's False.
- `df[df['cluster'] == 1]`: This is a DataFrame indexing operation using the Boolean series. It selects only the rows in df where the Boolean series is True, i.e., only the rows belonging to cluster 1. This operation effectively filters the DataFrame to contain only the data from cluster 1.